# Voice under Stress - Analysis

import of all functions needed

In [ ]:
import os
import sys

#from __future__ import division
import pandas as pd
import datetime as dt
import imageio
import numpy as np
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
import scipy 
#import librosa
import math
from statsmodels.stats.descriptivestats import sign_test

from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc, precision_recall_curve, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut, StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn import metrics, svm, linear_model
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler
from xgboost import XGBRegressor

from scipy import stats
from scipy.stats import skew, kurtosis
from statsmodels.stats.descriptivestats import sign_test

from matplotlib import rcParams

In [ ]:
homepath=os.getcwd()
scripts_path=os.path.join(homepath, 'scripts')
sys.path.append(scripts_path)

import importlib
import myml
import mystress
import myvoice

importlib.reload(myvoice)
importlib.reload(mystress)
importlib.reload(myml)

In [ ]:

# Check version of Python==64
!python -c "import sys; print(sys.maxsize > 2**32)"

# random seed for reproducability
np.random.seed(42)

# show all columns of dataframes
pd.set_option('display.max_columns', None)

# set current dir to highest hierachy to add data path
os.chdir('/')
data_folder='/data'

sys.path.append(data_folder)
os.chdir(homepath)


In [ ]:
filename_librosa='./processed_nemo_data/features.csv'
filename_behavior=data_folder + '/raw/TSST_behavior.csv'
filename_praat='./processed_nemo_data/new_praat_results'
filename_opensmile='./processed_nemo_data/opensmile_features.csv'
filename_output='./processed_nemo_data/df_vpn'

df, librosa_featurenames, praat_featurenames, opensimle_feature=mystress.load_df(filename_opensmile, filename_praat, 
                                                 filename_librosa, filename_behavior, filename_output)
df=mystress.calc_var(df)

In [ ]:
pd.set_option('display.max_columns', None)

# Main Code

### Step 1: load data and calculate variables

In [ ]:
df, librosa_featurenames, praat_featurenames, opensimle_feature=mystress.load_df(filename_opensmile, filename_praat, 
                                                 filename_librosa, filename_behavior, filename_output)
df=mystress.calc_var(df)

# In this Notebook the Training will only on the TSST condition data

In [ ]:
df_TSST = df[df["Cond"] == 1]

In [ ]:
len(opensimle_feature)

In [ ]:
audio=praat_featurenames +librosa_featurenames[0:40]+list(opensimle_feature)
audio.append('Sex')

In [ ]:
small_audio=['meanF0Hz', 'stdevF0Hz', 'HNR', 'localJitter', 'localShimmer', 'energy', 'Sex']

In [ ]:
#update nachänderung
import importlib
import myvoice
import myml

importlib.reload(myvoice)
importlib.reload(mystress)
importlib.reload(myml)

# Regression

## Random Forest Regressor

### with hyperparameter tuning (max_depth)

Here we are using a LOO (Leave One Out) for regression with a nested 5-fold CV for hyperparameter tuning (GridSearchCV).
In each iteraton the model is tuned on the training data and then predicts the hold out sample. Performance is compared to a mean-baseline and repeated for all folds.

This is done to predict Cortisol_React, cort_20_min, sAA_React, PANA_Delta_NA, PANA_Delta_PA

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'Cortisol_React')

space = dict()
space['max_depth'] = [2, 4, 5, 10]
model=RandomForestRegressor(n_estimators=1000) 
  
y_pred, y_true, z_test = myml.loo_regression(
    X=X,
    y=y,
    z=vpn,
    model=model,
    name="RF_tsst_cortisol_react",
    space=space,
    scaling="yes",
    features=audio
)


In [ ]:
myml.reg_cor(y_pred, y_true, 'RF', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'cort_20_min') 

space = dict()
space['max_depth'] = [2, 4, 5, 10]
model=RandomForestRegressor(n_estimators=1000) 
  
y_pred, y_true, z_test = myml.loo_regression(
    X=X,
    y=y,
    z=vpn,
    model=model,
    name="RF_cort_20_min",
    space=space,
    scaling="yes",
    features=audio
)

In [ ]:
myml.reg_cor(y_pred, y_true, 'RF', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'sAA_React')

space = dict()
space['max_depth'] = [2, 4, 5, 10]
model=RandomForestRegressor(n_estimators=1000) 

y_pred, y_true, z_test = myml.loo_regression(
    X=X,
    y=y,
    z=vpn,
    model=model,
    name="RF_saa_react",
    space=space,
    scaling="yes",
    features=audio
)

In [ ]:
myml.reg_cor(y_pred, y_true, 'RF', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'PANA_Delta_NA') 

space = dict()
space['max_depth'] = [2, 4, 5, 10]
model=RandomForestRegressor(n_estimators=1000) 
  
y_pred, y_true, z_test = myml.loo_regression(
    X=X,
    y=y,
    z=vpn,
    model=model,
    name="RF_pana_delta_na",
    space=space,
    scaling="yes",
    features=audio
)

In [ ]:
myml.reg_cor(y_pred, y_true, 'RF', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'PANA_Delta_PA') 

space = dict()
space['max_depth'] = [2, 4, 5, 10]
model=RandomForestRegressor(n_estimators=1000) 
  
y_pred, y_true, z_test = myml.loo_regression(
    X=X,
    y=y,
    z=vpn,
    model=model,
    name="RF_pana_delta_pa",
    space=space,
    scaling="yes",
    features=audio
)

In [ ]:
myml.reg_cor(y_pred, y_true, 'RF', 'true')

# Support Vector Machine

### with hyperparameter tuning (Kernel, Epsilon, C)

Here we are using a LOO (Leave One Out) for regression with a nested 5-fold CV for hyperparameter tuning (GridSearchCV).
In each iteraton the model is tuned on the training data and then predicts the hold out sample. Performance is compared to a mean-baseline and repeated for all folds.

This is done to predict Cortisol_React, cort_20_min, sAA_React, PANA_Delta_NA, PANA_Delta_PA

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'Cortisol_React') 

model = SVR(kernel='rbf', gamma='scale')

space = dict()
space['C'] = [10, 1.0, 0.1, 0.01]
space['kernel'] = ['linear', 'rbf']
space['gamma'] = ['scale', 0.001, 0.01, 0.1, 1, 10]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "SVR_cortisol_react", space, 'yes', audio)

In [ ]:
myml.reg_cor(y_pred, y_true, 'SVM', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'cort_20_min') 

model = SVR(kernel='rbf', gamma='scale')

space = dict()
space['C'] = [10, 1.0, 0.1, 0.01]
space['kernel'] = ['linear', 'rbf']
space['gamma'] = ['scale', 0.001, 0.01, 0.1, 1, 10]


y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "SVR_cort_20_min", space, 'yes', audio)

In [ ]:
myml.reg_cor(y_pred, y_true, 'SVM', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'sAA_React') 

model = SVR(kernel='rbf', gamma='scale')

space = dict()
space['C'] = [10, 1.0, 0.1, 0.01]
space['kernel'] = ['linear', 'rbf']
space['gamma'] = ['scale', 0.001, 0.01, 0.1, 1, 10]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "SVR_saa_react", space, 'yes', audio)

In [ ]:
myml.reg_cor(y_pred, y_true, 'SVM', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'PANA_Delta_NA') 

model = SVR(kernel='rbf', gamma='scale')

space = dict()
space['C'] = [10, 1.0, 0.1, 0.01]
space['kernel'] = ['linear', 'rbf']
space['gamma'] = ['scale', 0.001, 0.01, 0.1, 1, 10]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "SVR_pana_delta_na", space, 'yes', audio)

In [ ]:
myml.reg_cor(y_pred, y_true, 'SVM', 'true')

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'PANA_Delta_PA') 

model = SVR(kernel='rbf', gamma='scale')

space = dict()
space['C'] = [10, 1.0, 0.1, 0.01]
space['kernel'] = ['linear', 'rbf']
space['gamma'] = ['scale', 0.001, 0.01, 0.1, 1, 10]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "SVR_pana_delta_pa", space, 'yes', audio)

In [ ]:
myml.reg_cor(y_pred, y_true, 'SVM', 'true')

# XGBoost

### with hyperparameter tuning (max_depth, learning_rate, n_estimators)

Here we are using a LOO (Leave One Out) for regression with a nested 5-fold CV for hyperparameter tuning (GridSearchCV).
In each iteraton the model is tuned on the training data and then predicts the hold out sample. Performance is compared to a mean-baseline and repeated for all folds.

This is done to predict Cortisol_React, cort_20_min, sAA_React, PANA_Delta_NA, PANA_Delta_PA

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'Cortisol_React')

model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_jobs=-1
)

space = dict()
space["max_depth"] = [1, 2, 4, 8]
space["learning_rate"] = [0.03, 0.1, 0.2]
space["n_estimators"] = [50, 100, 150]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "XGB_cortisol_react", space, 'yes', audio)

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'cort_20_min')

model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_jobs=-1
)

space = dict()
space["max_depth"] = [1, 2, 4, 8]
space["learning_rate"] = [0.03, 0.1, 0.2]
space["n_estimators"] = [50, 100, 150]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "XGB_cort_20_min", space, 'yes', audio)

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'sAA_React')

model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_jobs=-1
)

space = dict()
space["max_depth"] = [1, 2, 4, 8]
space["learning_rate"] = [0.03, 0.1, 0.2]
space["n_estimators"] = [50, 100, 150]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "XGB_saa_react", space, 'yes', audio)

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'PANA_Delta_NA')

model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_jobs=-1
)

space = dict()
space["max_depth"] = [1, 2, 4, 8]
space["learning_rate"] = [0.03, 0.1, 0.2]
space["n_estimators"] = [50, 100, 150]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "XGB_pana_delta_na", space, 'yes', audio)

In [ ]:
X, y, vpn= myml.split_and_shuffle(df_TSST, audio, 'PANA_Delta_PA')

model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_jobs=-1
)

space = dict()
space["max_depth"] = [1, 2, 4, 8]
space["learning_rate"] = [0.03, 0.1, 0.2]
space["n_estimators"] = [50, 100, 150]

y_pred, y_true, z_test = myml.loo_regression(X, y, vpn, model, "XGB_pana_delta_pa", space, 'yes', audio)